# Neural Network + FGM Evasion (CSV Pipeline)

Flow implemented to match your scaffold:
1. Train a **clean NN** on clean training data.
2. Generate **FGM adversarial** samples from the clean model.
3. Build **C+A** dataset by concatenating clean + adversarial samples.
4. Retrain a second NN on **C+A** (retrained C+A model).
5. Compare clean model vs retrained C+A model on clean/adversarial (and C+A) evaluation sets.


Also includes an additional **AdversarialTrainer** branch for comparison.


In [149]:
# If needed, install once:
# !pip install torch scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [150]:
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEFAULT_DATA_PATH = Path(r"CSVs\newDataset.csv")
RUNS_DIR = Path(r"StandardizedRuns")
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - NN_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("NN_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"

# Columns that are not model features in this dataset
DROP_COLS = {LABEL_COL, "segment", "train", "sampling"}

TEST_SIZE = 0.75
BATCH_SIZE = 128
NB_EPOCHS = 25
LR = 1e-3
FGM_EPS = 0.10


def resolve_data_path() -> Path:
    if ENV_RUN_PATH:
        candidate = Path(ENV_RUN_PATH)
        if candidate.exists():
            return candidate
        print(f"[warn] Env path not found: {candidate}")

    if RUNS_DIR.exists():
        candidates = sorted(
            RUNS_DIR.glob(RUN_GLOB),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            return candidates[0]

    return DEFAULT_DATA_PATH


DATA_PATH = resolve_data_path()
print(f"Using data source: {DATA_PATH}")

EVAL_ON_CONCAT_TEST = True

ADV_RATIO = 0.2
WARMUP_EPOCHS = 8


Using data source: StandardizedRuns\NeuralNet_train_clean.csv


In [151]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols].to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(DATA_PATH))


Loaded: StandardizedRuns\NeuralNet_train_clean.csv
Rows=1698, Features=18, Label dist=[1351  347]
Train=(424, 18), Test=(1274, 18)


In [152]:
class MLP(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_art_classifier(d_in: int, lr: float = LR, class_weights: np.ndarray | None = None):
    model = MLP(d_in)

    if class_weights is None:
        criterion = nn.CrossEntropyLoss()
    else:
        cw = torch.tensor(class_weights, dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.Adam(model.parameters(), lr=lr)

    return PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


In [153]:
def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    probs = art_clf.predict(X)
    y_pred = np.argmax(probs, axis=1)

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y, y_pred))
    print(classification_report(y, y_pred, digits=4))

    return {"model_eval": name, "acc": acc, "f1": f1}


In [ ]:
# 1) Train clean model
art_clean = make_art_classifier(d_in=X_train.shape[1])
art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

# 2) Generate adversarial data from clean model (FGM)
fgm_clean = FastGradientMethod(estimator=art_clean, eps=FGM_EPS)
X_test_adv = fgm_clean.generate(x=X_test)
X_train_adv = fgm_clean.generate(x=X_train)

# Optional combined evaluation set C+A
X_train_ca = np.concatenate([X_train, X_train_adv], axis=0)
X_test_ca = np.concatenate([X_test, X_test_adv], axis=0)
#y_test_ca = np.concatenate([y_test, y_test], axis=0)

# Clean model evaluations
clean_on_clean = eval_classifier(art_clean, X_test, y_test, "clean_model_on_clean")
clean_on_adv = eval_classifier(art_clean, X_test_adv, y_test, "clean_model_on_fgm")

if EVAL_ON_CONCAT_TEST:
    clean_on_ca = eval_classifier(art_clean, X_test_ca, y_test_ca, "clean_model_on_C+A")
else:
    clean_on_ca = None



[clean_model_on_clean] acc=0.8391 f1=0.3533
confusion matrix:
[[1013    1]
 [ 204   56]]
              precision    recall  f1-score   support

           0     0.8324    0.9990    0.9081      1014
           1     0.9825    0.2154    0.3533       260

    accuracy                         0.8391      1274
   macro avg     0.9074    0.6072    0.6307      1274
weighted avg     0.8630    0.8391    0.7949      1274


[clean_model_on_fgm] acc=0.8430 f1=0.4624
confusion matrix:
[[988  26]
 [174  86]]
              precision    recall  f1-score   support

           0     0.8503    0.9744    0.9081      1014
           1     0.7679    0.3308    0.4624       260

    accuracy                         0.8430      1274
   macro avg     0.8091    0.6526    0.6852      1274
weighted avg     0.8334    0.8430    0.8171      1274


[clean_model_on_C+A] acc=0.8411 f1=0.4122
confusion matrix:
[[2001   27]
 [ 378  142]]
              precision    recall  f1-score   support

           0     0.8411    0.

In [155]:
# 3) Retrain on concatenated clean + adversarial (C+A)
X_train_concat = np.concatenate([X_train, X_train_adv], axis=0)
y_train_concat = np.concatenate([y_train, y_train], axis=0)

perm = np.random.permutation(len(y_train_concat))
X_train_concat, y_train_concat = X_train_concat[perm], y_train_concat[perm]

art_retrained_ca = make_art_classifier(d_in=X_train.shape[1])
art_retrained_ca.fit(X_train_concat, y_train_concat, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

# Retrained C+A model evaluations
retrained_on_clean = eval_classifier(art_retrained_ca, X_test, y_test, "retrained_C+A_on_clean")
retrained_on_adv = eval_classifier(art_retrained_ca, X_test_adv, y_test, "retrained_C+A_on_fgm")

if EVAL_ON_CONCAT_TEST:
    retrained_on_ca = eval_classifier(art_retrained_ca, X_test_ca, y_test_ca, "retrained_C+A_on_C+A")
else:
    retrained_on_ca = None



[retrained_C+A_on_clean] acc=0.8438 f1=0.3839
confusion matrix:
[[1013    1]
 [ 198   62]]
              precision    recall  f1-score   support

           0     0.8365    0.9990    0.9106      1014
           1     0.9841    0.2385    0.3839       260

    accuracy                         0.8438      1274
   macro avg     0.9103    0.6187    0.6472      1274
weighted avg     0.8666    0.8438    0.8031      1274


[retrained_C+A_on_fgm] acc=0.8383 f1=0.3941
confusion matrix:
[[1001   13]
 [ 193   67]]
              precision    recall  f1-score   support

           0     0.8384    0.9872    0.9067      1014
           1     0.8375    0.2577    0.3941       260

    accuracy                         0.8383      1274
   macro avg     0.8379    0.6224    0.6504      1274
weighted avg     0.8382    0.8383    0.8021      1274


[retrained_C+A_on_C+A] acc=0.8411 f1=0.3891
confusion matrix:
[[2014   14]
 [ 391  129]]
              precision    recall  f1-score   support

           0     0.

In [156]:
# 4) AdversarialTrainer branch (kept as requested)
# Use class-weighted loss + clean warm-start to reduce majority-class collapse.
class_counts = np.bincount(y_train)
class_weights = (len(y_train) / (2.0 * class_counts)).astype(np.float32)
print(f"Class counts: {class_counts}, class weights: {class_weights}")

art_adv = make_art_classifier(d_in=X_train.shape[1], class_weights=class_weights)

# Warm-start on clean data
art_adv.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=WARMUP_EPOCHS)

fgm_for_training = FastGradientMethod(estimator=art_adv, eps=FGM_EPS)
trainer = AdversarialTrainer(classifier=art_adv, attacks=[fgm_for_training], ratio=ADV_RATIO)
trainer.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

art_adv = trainer.get_classifier()

adv_on_clean = eval_classifier(art_adv, X_test, y_test, "adv_trained_on_clean")
adv_on_adv = eval_classifier(art_adv, X_test_adv, y_test, "adv_trained_on_fgm")

if EVAL_ON_CONCAT_TEST:
    adv_on_ca = eval_classifier(art_adv, X_test_ca, y_test_ca, "adv_trained_on_C+A")
else:
    adv_on_ca = None


Class counts: [337  87], class weights: [0.6290801 2.4367816]


Adversarial training epochs: 100%|██████████| 25/25 [00:00<00:00, 75.18it/s]


[adv_trained_on_clean] acc=0.8846 f1=0.6621
confusion matrix:
[[983  31]
 [116 144]]
              precision    recall  f1-score   support

           0     0.8944    0.9694    0.9304      1014
           1     0.8229    0.5538    0.6621       260

    accuracy                         0.8846      1274
   macro avg     0.8587    0.7616    0.7962      1274
weighted avg     0.8798    0.8846    0.8757      1274


[adv_trained_on_fgm] acc=0.3666 f1=0.3872
confusion matrix:
[[212 802]
 [  5 255]]
              precision    recall  f1-score   support

           0     0.9770    0.2091    0.3444      1014
           1     0.2412    0.9808    0.3872       260

    accuracy                         0.3666      1274
   macro avg     0.6091    0.5949    0.3658      1274
weighted avg     0.8268    0.3666    0.3532      1274


[adv_trained_on_C+A] acc=0.6256 f1=0.4555
confusion matrix:
[[1195  833]
 [ 121  399]]
              precision    recall  f1-score   support

           0     0.9081    0.5893

In [157]:
rows = [
    clean_on_clean,
    clean_on_adv,
    retrained_on_clean,
    retrained_on_adv,
    adv_on_clean,
    adv_on_adv,
]

if EVAL_ON_CONCAT_TEST and clean_on_ca is not None and retrained_on_ca is not None:
    rows.extend([clean_on_ca, retrained_on_ca])

if EVAL_ON_CONCAT_TEST and adv_on_ca is not None:
    rows.append(adv_on_ca)

summary_df = pd.DataFrame(rows)
summary_df


,model_eval,acc,f1
0,clean_model_on_clean,0.839089,0.353312
1,clean_model_on_fgm,0.843014,0.462366
2,retrained_C+A_on_clean,0.843799,0.383901
3,retrained_C+A_on_fgm,0.838305,0.394118
4,adv_trained_on_clean,0.884615,0.662069
5,adv_trained_on_fgm,0.366562,0.387244
6,clean_model_on_C+A,0.841052,0.412192
7,retrained_C+A_on_C+A,0.841052,0.389140
8,adv_trained_on_C+A,0.625589,0.455479


In [158]:
# Optional: save metrics
out_csv = r"Results\NeuralNetworksResults\nn_fgm_evasion_summary.csv"
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: Results\NeuralNetworksResults\nn_fgm_evasion_summary.csv
